In [1]:
from typing import Literal
from langgraph.types import Command
from langgraph.graph import StateGraph, MessagesState, START, END
from langchain_core.messages import AIMessage

def agent_1(state: MessagesState):
    messages = AIMessage(content='This is a message from agent 1.')
    return Command(
        goto='agent_2',
        update={'messages': [messages]}
    )

def agent_2(state:MessagesState):
    message = AIMessage(content='This is a message from agent 2.')
    return Command(
        goto=END,
        update={'messages': [message]}
    )

graph_builder = StateGraph(MessagesState)
graph_builder.add_node('agent_1', agent_1)
graph_builder.add_node('agent_2', agent_2)

graph_builder.add_edge(START, 'agent_1')

graph = graph_builder.compile()

In [2]:
for chunk in graph.stream({'messages': []}, stream_mode='values'):
    print(chunk)

{'messages': []}
{'messages': [AIMessage(content='This is a message from agent 1.', additional_kwargs={}, response_metadata={}, id='7dffe837-e773-4ad0-a217-8298e38ee30b')]}
{'messages': [AIMessage(content='This is a message from agent 1.', additional_kwargs={}, response_metadata={}, id='7dffe837-e773-4ad0-a217-8298e38ee30b'), AIMessage(content='This is a message from agent 2.', additional_kwargs={}, response_metadata={}, id='67b37f87-a4aa-4a95-888b-58e85969f6fd')]}


In [3]:
def node2(state):
    return Command(
        goto='AgentB',
        update={'my_state_key': 'my_state_value'},
        graph=Command.PARENT,
    )

In [4]:
from langchain_core.tools import tool

@tool
def transfer_to_book():
    '''Transfer to book.'''
    return Command(
        goto='book',
        update={'my_state_key': 'my_state_value'},
        graph=Command.PARENT,
    )